In [23]:
import torch 
import torch.nn as nn
import torch as th
import torch.nn.functional as F
from transformers import GPT2Config, GPT2Model, GPT2Tokenizer, GPT2LMHeadModel

In [ ]:


# Initialize GPT-2 configuration
config = GPT2Config(
    vocab_size=2,
    n_positions=40,
    n_embd=384,
    n_layer=6,
    n_head=6,
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
    loss_type="ForCausalLMLoss"
)

# Create GPT-2 model (without LM head)
model = GPT2Model(config)
model_lm = GPT2LMHeadModel(config)

# Initialize tokenizer
# tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model_lm.to(device)

print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Using device: {device}")


Model initialized with 10,663,680 parameters
Using device: cuda


In [21]:
outputs.logits.shape

torch.Size([1, 6, 2])

In [ ]:
with th.no_grad():
    # Example of computing loss for next token prediction
    input_ids = th.tensor([[0,1,0,1,1,0]]).cuda()
    outputs = model_lm(input_ids)
    loss = F.cross_entropy(outputs.logits[:, :-1, :].permute(0,2,1), input_ids[:, 1:])
    print(loss)

tensor(1.0984, device='cuda:0')


### Training loop

In [31]:
import sys
sys.path.append("/n/home12/binxuwang/Github/DiffusionAttnConsistency")
from core.parity_lib import sample_ensuring_uniqueness, sample_group_parity_vec

# Example 1: Generate 1000 unique samples with group parity constraints
N = 4096
sample_len = 36  # total vector length
group_size = 9  # each group of 12 bits must have even parity

# Define your sampling function
def my_sample_func(N):
    return sample_group_parity_vec(N=N, sample_len=sample_len,
                                   group_size=group_size, parity=0)

# Generate unique samples
train_samples = sample_ensuring_uniqueness(N=N, sample_func=my_sample_func)
train_samples_01 = (train_samples + 1) // 2
train_samples_01 = train_samples_01.astype(int)

In [32]:
train_samples_01.shape

(4096, 36)

In [40]:
# Initialize GPT-2 configuration
config = GPT2Config(
    vocab_size=3,
    n_positions=40,
    n_embd=384,
    n_layer=6,
    n_head=6,
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
    loss_type="ForCausalLMLoss"
)

# Create GPT-2 model (without LM head)
# model = GPT2Model(config)
model_lm = GPT2LMHeadModel(config)
model_lm.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(3, 384)
    (wpe): Embedding(40, 384)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=1152, nx=384)
          (c_proj): Conv1D(nf=384, nx=384)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=1536, nx=384)
          (c_proj): Conv1D(nf=384, nx=1536)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=384, out_features=3, bias=False)
)

In [53]:
loss = F.cross_entropy(outputs.logits[:, :-1, :].permute(0,2,1), input_ids[:, 1:])
th.allclose(loss, outputs.loss)

True

In [57]:
def sample_sequences(model, num_samples=10, max_length=36, sos_token=2, temperature=1.0):
    """Sample sequences from the trained GPT model in parallel"""
    with torch.no_grad():
        # Start with SOS token for all samples
        input_ids = torch.full((num_samples, 1), sos_token, dtype=torch.long).cuda()
        
        # Generate sequence token by token for all samples simultaneously
        for step in range(max_length):
            # Get model predictions for all samples
            outputs = model(input_ids)
            logits = outputs.logits
            
            # Apply temperature and sample next token for all samples
            next_token_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_tokens = torch.multinomial(probs, 1)
            
            # Append to all sequences
            input_ids = torch.cat([input_ids, next_tokens], dim=1)
        
        # Remove SOS token and convert to numpy
        sequences = input_ids[:, 1:].cpu()#.numpy()  # Remove SOS token
    
    return sequences



In [64]:
N = 4096
sample_len = 36  # total vector length
group_size = 6  # each group of 12 bits must have even parity
num_groups = sample_len // group_size
expected_parity = 0  # even parity

# Generate unique samples
train_samples = sample_ensuring_uniqueness(N=N, 
                    sample_func=lambda N: sample_group_parity_vec(N=N, sample_len=sample_len,
                                   group_size=group_size, parity=expected_parity))
train_samples_01 = (train_samples + 1) // 2
train_samples_01 = train_samples_01.astype(int)

train_samples_01_sos = th.cat([2 * th.ones(train_samples_01.shape[0], 1, dtype=th.long), 
                               th.tensor(train_samples_01, dtype=th.long)], dim=1)
train_samples_01_sos.shape

torch.Size([4096, 37])

In [72]:
# Training loop for GPT model on parity data
from core.parity_lib import parity_func
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
eval_sample_num = 1024
# Convert training data to tensors
train_tensor = torch.tensor(train_samples_01_sos, dtype=torch.long).cuda()
# Create dataset and dataloader
dataset = TensorDataset(train_tensor)
batch_size = 256
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
# Setup optimizer
optimizer = optim.AdamW(model_lm.parameters(), lr=1e-4, weight_decay=0.01)
# Training parameters
num_epochs = 500
print_every = 50
eval_every = 50
# Training loop
model_lm.train()
total_loss = 0
step = 0
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch_idx, (batch_data,) in enumerate(dataloader):
        optimizer.zero_grad()
        # Prepare input and labels for next token prediction
        input_ids = batch_data.to(device)
        # labels = input_ids.clone()
        # Forward pass
        outputs = model_lm(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss
        # Backward pass
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        epoch_loss += loss.item()
        step += 1
        if step % print_every == 0:
            avg_loss = total_loss / step
            print(f"Step {step}, Avg Loss: {avg_loss:.4f}, Current Loss: {loss.item():.4f}")

        if step % eval_every == 0:
            model_lm.eval()
            generated_samples = sample_sequences(model_lm, num_samples=eval_sample_num, 
                                                        max_length=sample_len, sos_token=2, temperature=1.0)
            generated_samples_parity = generated_samples * 2 - 1
            with th.no_grad():
                sample_pergroup_int = generated_samples_parity.reshape(eval_sample_num, num_groups, group_size)
            sample_eval_parity = parity_func(sample_pergroup_int, axis=-1)
            # count parity correctness for each group
            pergroup_parity_correctness = th.sum(sample_eval_parity == expected_parity).sum()
            pergroup_parity_correctness_ratio = pergroup_parity_correctness / (eval_sample_num * num_groups)
            # count sample correct only when all groups are correct
            sample_correctness = th.all(sample_eval_parity == expected_parity, dim=1).sum()
            sample_correctness_ratio = sample_correctness / eval_sample_num
            print(f"pergroup_parity_acc: {pergroup_parity_correctness_ratio} [{pergroup_parity_correctness}/{eval_sample_num * num_groups}], "+\
                f"sample_acc: {sample_correctness_ratio} [{sample_correctness}/{eval_sample_num}]")
            model_lm.train()
        
    avg_epoch_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_epoch_loss:.4f}")
    
print("Training completed!")
print(f"Final average loss: {total_loss / step:.4f}")

/tmp/ipykernel_808853/364138210.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_tensor = torch.tensor(train_samples_01_sos, dtype=torch.long).cuda()


Epoch 1/500, Average Loss: 0.6335
Epoch 2/500, Average Loss: 0.6328
Epoch 3/500, Average Loss: 0.6318
Step 50, Avg Loss: 0.6326, Current Loss: 0.6309
pergroup_parity_acc: 0.7506510615348816 [4612/6144], sample_acc: 0.1025390625 [105/1024]
Epoch 4/500, Average Loss: 0.6308
Epoch 5/500, Average Loss: 0.6307
Epoch 6/500, Average Loss: 0.6309
Step 100, Avg Loss: 0.6317, Current Loss: 0.6296
pergroup_parity_acc: 0.7454426884651184 [4580/6144], sample_acc: 0.1181640625 [121/1024]
Epoch 7/500, Average Loss: 0.6312
Epoch 8/500, Average Loss: 0.6303
Epoch 9/500, Average Loss: 0.6304
Step 150, Avg Loss: 0.6313, Current Loss: 0.6296
pergroup_parity_acc: 0.7431640625 [4566/6144], sample_acc: 0.107421875 [110/1024]
Epoch 10/500, Average Loss: 0.6300
Epoch 11/500, Average Loss: 0.6300
Epoch 12/500, Average Loss: 0.6304
Step 200, Avg Loss: 0.6310, Current Loss: 0.6292
pergroup_parity_acc: 0.7566731572151184 [4649/6144], sample_acc: 0.134765625 [138/1024]
Epoch 13/500, Average Loss: 0.6298
Epoch 14/50

In [68]:
from scripts.parity_memorization_eval_cli import compute_membership_counts

In [70]:
sample_mem_stats = compute_membership_counts(th.tensor(train_samples).flatten(start_dim=1).cpu(), 
                                             generated_samples_parity.cpu(), group_size)
print(sample_mem_stats)

{'sample_mem_num': 0, 'sample_mem_ratio': 0.0, 'bitgroup_mem_num': 4573, 'bitgroup_mem_ratio': 0.7443033854166666}


In [63]:
from core.parity_lib import parity_func
group_size = 9
num_groups = sample_len // group_size
expected_parity = 0  # even parity
eval_sample_num = 1024
generated_samples = sample_sequences(model_lm, num_samples=eval_sample_num, 
                                            max_length=sample_len, sos_token=2, temperature=1.0)
generated_samples_parity = generated_samples * 2 - 1
sample_pergroup_int = generated_samples_parity.reshape(eval_sample_num, num_groups, group_size)
sample_eval_parity = parity_func(sample_pergroup_int, axis=-1)
# count parity correctness for each group
pergroup_parity_correctness = th.sum(sample_eval_parity == expected_parity).sum()
pergroup_parity_correctness_ratio = pergroup_parity_correctness / (eval_sample_num * num_groups)
# count sample correct only when all groups are correct
sample_correctness = th.all(sample_eval_parity == expected_parity, dim=1).sum()
sample_correctness_ratio = sample_correctness / eval_sample_num
print(f"pergroup_parity_acc: {pergroup_parity_correctness_ratio} [{pergroup_parity_correctness}/{eval_sample_num * num_groups}], sample_acc: {sample_correctness_ratio} [{sample_correctness}/{eval_sample_num}]")

pergroup_parity_acc: 0.6181640625 [2532/4096], sample_acc: 0.119140625 [122/1024]


In [ ]:

# Sample from the trained model (parallel version for speed)
model_lm.eval()
# Generate samples (much faster now!)
print("Sampling from trained model...")
num_samples = 512
generated_samples = sample_sequences(model_lm, num_samples=num_samples, 
                                            max_length=sample_len, sos_token=2, temperature=0.8)

print(f"Generated {len(generated_samples)} sequences of length {generated_samples.shape[1]}")
print("\nFirst 5 generated sequences:")
for i in range(min(5, len(generated_samples))):
    print(f"Sample {i+1}: {generated_samples[i]}")

# Check parity satisfaction
from core.parity_lib import parity_func
# Convert back to {-1, +1} format for parity checking
generated_samples_parity = generated_samples * 2 - 1
# Check group-wise parity
group_size = 9
num_groups = sample_len // group_size
parity_violations = 0
expected_parity = 0  # even parity
print(f"\nChecking parity constraints (group_size={group_size}):")
for i, sample in enumerate(generated_samples_parity):
    groups = sample.reshape(num_groups, group_size)
    group_parities = parity_func(groups, axis=1) # [parity_func(group) for group in groups]
    
    # Check if all groups have even parity (parity=0)
    violations = sum(1 for p in group_parities if p != expected_parity)
    parity_violations += violations
    
    if i < 3:  # Show details for first 3 samples
        print(f"Sample {i+1} group parities: {group_parities}, violations: {violations}")

print(f"\nTotal parity violations: {parity_violations} out of {num_samples * num_groups} groups")
print(f"Parity satisfaction rate: {(1 - parity_violations/(num_samples * num_groups)):.3f}")

# Compare with training data statistics
print(f"\nTraining data shape: {train_samples_01.shape}")
print(f"Generated data shape: {generated_samples.shape}")

Sampling from trained model...
Generated 512 sequences of length 36

First 5 generated sequences:
Sample 1: [1 1 1 1 1 0 0 1 1 0 1 0 1 1 0 1 0 1 0 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 1 1]
Sample 2: [1 0 0 0 1 1 0 0 0 0 0 1 1 1 0 0 1 0 1 0 0 0 1 0 0 1 0 0 1 1 0 1 0 1 1 1]
Sample 3: [1 1 0 0 0 0 0 0 1 1 0 0 1 1 1 0 0 0 0 1 1 0 0 1 1 1 0 1 1 1 0 0 1 0 0 1]
Sample 4: [1 0 1 1 0 1 0 1 0 0 0 0 1 1 1 0 0 1 0 0 0 0 0 0 0 1 0 1 0 1 1 1 1 1 1 0]
Sample 5: [1 1 1 1 0 1 1 0 1 0 0 0 1 0 1 1 1 1 1 0 1 0 0 0 1 1 0 0 0 0 0 1 0 1 1 1]

Checking parity constraints (group_size=9):
Sample 1 group parities: [0 0 1 1], violations: 2
Sample 2 group parities: [0 1 0 1], violations: 2
Sample 3 group parities: [0 1 0 0], violations: 1

Total parity violations: 785 out of 2048 groups
Parity satisfaction rate: 0.617

Training data shape: (4096, 36)
Generated data shape: (512, 36)


In [8]:
import torch as th
with th.no_grad():
    outputs = model(th.tensor([[0,1,0,1,1,0]]).cuda())
outputs

BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=tensor([[[ 0.6863,  0.6001,  0.6132,  ..., -0.5295,  0.3739, -0.6264],
         [-0.2593,  0.7309,  0.8888,  ..., -1.8026, -0.7796, -1.1115],
         [-0.3673,  1.1051, -0.0626,  ...,  0.2213, -0.5473, -1.4563],
         [-1.0522,  0.8073, -0.4339,  ..., -0.5230, -0.5663, -0.6196],
         [ 0.9497,  0.7785,  0.2382,  ...,  0.0833, -1.4332, -1.0499],
         [ 0.6275,  1.4233,  1.1630,  ..., -1.1253, -0.1861,  0.0210]]],
       device='cuda:0'), past_key_values=((tensor([[[[ 0.2962,  0.4112, -0.0110,  ...,  0.4239,  0.0725, -0.4406],
          [ 0.3930,  0.5909, -0.0372,  ...,  0.0489,  0.0729, -0.5545],
          [-0.0084,  0.3599,  0.2217,  ...,  0.6391,  0.4081,  0.0018],
          [ 0.2434,  0.5157,  0.5316,  ..., -0.3571,  0.0242,  0.2198],
          [ 0.1863,  0.4121,  0.2063,  ...,  0.4853, -0.0113, -0.1576],
          [-0.1365,  0.3726,  0.1835,  ..., -0.0869,  0.0772,  0.1201]],

         [[ 0.3391,  0.0749,  0.634

In [ ]:
import torch as th


class ParityGPT2Model(nn.Module):
    def __init__(self, vocab_size=2, max_length=128, n_embd=768, n_class=0, **kwargs):
        super().__init__()
        # Combine embeddings
        combined_embedding_size = n_embd  # Adjust based on your combination strategy
        # if is_sep_embed:
        #     self.sep_word_embed = SepWordEmbed(attribute_dims, embed_size=n_embd//3)
        #     self.multi_lmhead = SepLMhead(attribute_dims, embed_size=n_embd//3)
        # else:
        #     self.sep_word_embed = CmbWordEmbed(attribute_dims, embed_size=n_embd)
        #     self.multi_lmhead = CmbLMhead(attribute_dims, embed_size=n_embd)
        config = GPT2Config(vocab_size=vocab_size, n_positions=max_length, n_embd=combined_embedding_size, **kwargs)
        self.gpt2 = GPT2Model(config)
        self.context_embed = nn.Embedding(1+n_class, n_embd)

    def forward(self, input_ids, y=None):
        # input_ids is expected to be a list of three tensors [attr1, attr2, attr3]
        if y is None:
            y = torch.zeros(input_ids.shape[0], dtype=th.long).to(input_ids[0].device)
        ctx_vec = self.context_embed(y)
        combined_embedding = self.sep_word_embed(input_ids)
        combined_embedding = torch.concat([ctx_vec[:,None,:], combined_embedding, ], dim=1)
        outputs = self.gpt2(inputs_embeds=combined_embedding)
        logits_attr1, logits_attr2, logits_attr3 = self.multi_lmhead(outputs.last_hidden_state)
        return outputs, logits_attr1, logits_attr2, logits_attr3

In [ ]:
import torch.nn.functional as F

def next_token_loss(outputs, targets, loss_fn=F.cross_entropy):
    logits1, logits2, logits3 = outputs[0], outputs[1], outputs[2]
    loss1 = loss_fn(logits1[:, :-1, :].permute(0,2,1), targets[:, 1:, 0])
    loss2 = loss_fn(logits2[:, :-1, :].permute(0,2,1), targets[:, 1:, 1])
    loss3 = loss_fn(logits3[:, :-1, :].permute(0,2,1), targets[:, 1:, 2])
    return loss1 + loss2 + loss3